# 기초 다지기 직접 해보기: 수학과 파이썬 열 단원

기초 다지기 주차의 정리 슬라이드 열 단원을 한 문제씩 손으로 옮겨 보는 연습이에요. 파이썬을 한 줄도 안 써 봤어도 괜찮아요.

- 문제마다 **생각 순서**가 먼저 나와요. 코드를 치기 전에 그 순서를 말로 한 번 읊어요.
- 여러 줄을 새로 쓰는 문제에는 **의사코드**가 따로 붙어 있어요. 의사코드 한 줄이 파이썬 한 줄이 돼요.
- `# TODO` 가 있는 칸의 `None` 을 알맞은 코드나 값으로 바꿔요.
- 바로 아래 **확인 셀**을 실행하면 맞았는지 알려 줘요. 틀리면 빨간 AssertionError 와 힌트가 나와요.
- 막히면 맨 아래 **정답 코드**를 봐요. 모범 의사코드도 거기 같이 있어요.

여기 나오는 숫자는 거의 다 정리 슬라이드의 손계산에 그대로 적혀 있어요. 답이 안 맞으면 그 슬라이드를 다시 봐요. 1번부터 7번까지는 파이썬만, 3번과 6번과 9번은 넘파이, 10번만 PyTorch 를 써요. GPU 도 다운로드도 필요 없어요.

In [ ]:
import math
from collections import Counter
import numpy as np
import torch
def ok(cond, msg):
    assert cond, msg
    print('정답! ' + msg)
print('준비 끝')

## 1. 벡터는 숫자 목록, 차원은 칸의 개수

기초 1단원. **벡터(Vector)** 는 숫자를 순서대로 적은 목록이고, 파이썬에서는 대괄호로 적어요. `a = [2, 1]` 이에요.

**차원(Dimension)** 은 칸이 몇 개인지, 곧 `len(a)` 예요. 더하기는 같은 자리끼리만 해요. **원-핫 벡터(One-hot Vector)** 는 자기 자리만 1 이고 나머지는 모두 0 인 목록이에요.

숫자는 슬라이드 손계산 1 과 손계산 3 에 그대로 있어요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 벡터는 리스트로 적어요. a = [2, 1] 이면 칸이 두 개예요
2. 차원은 칸의 개수예요. len(a) 가 바로 그 답이에요
3. 더하기는 첫 칸은 첫 칸끼리, 둘째 칸은 둘째 칸끼리예요. 자리가 섞이지 않아요
4. 원-핫 벡터는 단어 목록에서 그 단어의 번호 자리만 1 로 켜요
5. 모텔과 호텔을 같은 자리끼리 곱해서 더하면 0 이 나와야 해요

### 의사코드 (한 줄이 파이썬 한 줄이 돼요)

```
a 와 b 를 리스트로 적는다
차원 = a 의 칸 개수
a + b = [첫 칸끼리 더한 값, 둘째 칸끼리 더한 값]
모텔 벡터와 호텔 벡터를 같은 자리끼리 곱해서 모두 더한다
```

> **나중에 여기서 만나요** N2 p.23-25 에서 단어를 기호로만 다루면 **원-핫 벡터** 가 되고, p.25 슬라이드가 motel 과 hotel 의 값이 0 이라고 적어요. 그 한계 때문에 N2 p.26-27 의 **단어 벡터** 가 나오고, N4 p.40 에서는 그 임베딩 벡터에 위치 벡터를 더해서 트랜스포머에 넣어요.

In [ ]:
a = [2, 1]
b = [1, 3]

dim = None         # TODO: a 의 차원. 칸이 몇 개인지 (len 사용)
a_plus_b = None    # TODO: 같은 자리끼리 더한 리스트. [첫 칸 합, 둘째 칸 합]
two_a = [2 * v for v in a]

vocab = ['개', '고양이', '모텔', '호텔', '사과']

def onehot(word):
    i = vocab.index(word)
    return [1 if k == i else 0 for k in range(len(vocab))]

motel = onehot('모텔')
hotel = onehot('호텔')
same = None        # TODO: motel 과 hotel 을 같은 자리끼리 곱해서 모두 더하기 (sum 과 zip)

print(dim, a_plus_b, two_a)
print(motel, hotel, same)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(dim == 2, '(2, 1) 은 칸이 2개라 2차원')
ok(a_plus_b == [3, 4], '(2,1) + (1,3) = (3,4). 슬라이드 손계산 1 과 같아요')
ok(two_a == [4, 2], '스칼라 배는 칸마다 똑같이 곱해요')
ok(motel == [0, 0, 1, 0, 0] and hotel == [0, 0, 0, 1, 0], '원-핫 벡터는 자기 자리만 1')
ok(same == 0, '모텔과 호텔은 뜻이 비슷한데 값이 0. 원-핫 벡터는 비슷함을 전혀 못 담아요')

## 2. 내적, 길이, 코사인 유사도를 함수로

기초 2단원. **내적(Dot Product)** 은 같은 자리끼리 곱해서 모두 더한 값이고 답은 숫자 하나예요. **노름(Norm)** 은 칸마다 제곱해서 더한 뒤 루트를 씌운 길이예요. **코사인 유사도(Cosine Similarity)** 는 내적을 두 길이의 곱으로 나눈 값이에요.

마지막 칸에서는 찾는 벡터 하나를 후보 셋과 내적해서 점수를 매겨요. 이 점수가 4주차의 **어텐션 점수(Attention Score)** 예요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 내적은 자리마다 곱한 값을 모두 더하기예요. zip 이 두 리스트를 자리별로 짝지어 줘요
2. 노름은 자기 자신과의 내적에 루트예요. dot(u, u) 에 math.sqrt 를 씌우면 끝이에요
3. 코사인은 내적을 두 노름의 곱으로 나누기예요. 함수 두 개를 이미 만들어 뒀으니 한 줄이에요
4. 점수 매기기는 후보마다 내적 한 번이에요. 리스트 내포로 한 줄에 써요
5. 확인: (3,2)와 (1,4) 는 11, (3,4) 의 길이는 5, (3,4)와 (4,3) 의 코사인은 0.96

> **나중에 여기서 만나요** N2 p.47-48 에서 단어 벡터를 평가할 때 사람 점수와 코사인 유사도를 견주고, 단어 유추도 코사인이 가장 큰 단어를 골라요. N4 p.21 은 the simplest score is a dot product 라고 적어요. 여기 만든 `dot` 이 그 어텐션 점수이고, N4 p.38 에서 그 값을 루트 $d_k$ 로 나눠요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
dot(u, v) = 자리마다 u 와 v 를 곱한 값을 모두 더한다
norm(u) = dot(u, u) 에 루트를 씌운다
cosine(u, v) = dot(u, v) 를 norm(u) 곱하기 norm(v) 로 나눈다
점수 = 후보마다 dot(q, 후보)
가장 큰 점수의 자리 번호를 찾는다
```

In [ ]:
def dot(u, v):
    return None        # TODO: 같은 자리끼리 곱해서 모두 더하기 (sum 과 zip)

def norm(u):
    return None        # TODO: 자기 자신과의 내적에 루트 (math.sqrt)

def cosine(u, v):
    return None        # TODO: 내적을 두 노름의 곱으로 나누기

print(dot([3, 2], [1, 4]), norm([3, 4]), round(cosine([3, 4], [4, 3]), 4))

q = [1, 2]
keys = [[2, 0], [0, 3], [1, 1]]
scores = [dot(q, k) for k in keys]
best = scores.index(max(scores))
print(scores, best)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(dot([3, 2], [1, 4]) == 11, '3x1 + 2x4 = 11. 슬라이드 손계산 1 과 같아요')
ok(norm([3, 4]) == 5.0, '9 + 16 = 25, 루트를 씌우면 5')
ok(abs(cosine([3, 4], [4, 3]) - 0.96) < 1e-9, '24 나누기 25 = 0.96')
ok(abs(cosine([1, 0], [0, 1])) < 1e-12, '직각이면 코사인도 내적도 0')
ok(scores == [2, 6, 3], '어텐션 점수는 2, 6, 3')
ok(best == 1, '가장 큰 점수는 두 번째 후보. 이 고르기가 어텐션의 1단계예요')

## 3. shape 을 먼저 적고 나서 곱하기

기초 3단원. **행렬(Matrix)** 의 **모양(Shape)** 은 언제나 (행, 열) 순서예요. 곱셈 규칙은 `(a, b) @ (b, c) -> (a, c)` 하나예요. 가운데가 같아야 곱할 수 있고, 그 숫자는 사라져요.

코드를 돌리기 **전에** 답을 튜플로 적어요. 교수님이 3주차에 transpose 위치를 외우기보다 shape 을 맞춘다고 생각하라고 했어요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. A 는 (2, 3), x 는 (3,) 이에요. 가운데 3 이 같으니 곱할 수 있어요
2. 가운데 3 이 사라지고 바깥 2 만 남아요. 그래서 A @ x 는 (2,) 예요
3. Q 와 K 는 둘 다 (3, 2) 예요. 그냥 곱하면 2 와 3 이 달라서 안 돼요
4. K 를 전치하면 (2, 3) 이 되고, (3,2) @ (2,3) 은 (3,3) 이에요
5. 멀티 헤드는 전체 칸 수를 헤드 수로 나누는 것뿐이에요. 512 를 8 로 나눠요

### 의사코드 (한 줄이 파이썬 한 줄이 돼요)

```
A @ x 의 모양을 먼저 종이에 적는다: (2,3) 과 (3,) 이니 (2,)
실제로 곱해서 값을 본다
Q 와 K 의 모양을 적는다: 둘 다 (3,2)
K 를 전치해서 (2,3) 으로 만들고 Q 와 곱한다
헤드 하나의 칸 수 = 전체 칸 수 나누기 헤드 수
```

> **나중에 여기서 만나요** N3 p.25-26 이 전치와 shape 맞추기, 기울기의 shape 은 파라미터와 같아야 한다는 쪽이에요. N4 p.37 에서 $Q @ K^{\top}$ 가 만드는 $(n, n)$ 점수 행렬이 여기 만든 `S` 이고, N4 p.47 의 멀티 헤드가 $d = 512$ 를 $h = 8$ 로 나눠 64 칸씩 쓰는 그 나눗셈이에요. 과제 2(N3 p.57)에서 word2vec 기울기를 손으로 구할 때 첫 번째 검산도 이 shape 확인이에요.

In [ ]:
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
x = np.array([1.0, 0.0, 2.0])

Ax = A @ x
my_Ax_shape = None     # TODO: 돌리기 전에 튜플로 적어요. 칸이 하나면 (7,) 처럼 써요

Q = np.array([[2.0, 0.0],
              [0.0, 2.0],
              [1.0, 1.0]])
K = Q.copy()
S = None               # TODO: Q 와 K 의 전치를 곱하기 (K.T 사용)
my_S_shape = None      # TODO: S 의 모양

d, h = 512, 8
head_dim = None        # TODO: 헤드 하나가 쓰는 칸 수 (// 사용)

print(Ax, tuple(S.shape), head_dim)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(np.allclose(Ax, [7.0, 16.0]), 'A @ x = (7, 16). 줄마다 내적 한 번씩이에요')
ok(tuple(Ax.shape) == my_Ax_shape == (2,), '(2,3) @ (3,) 은 (2,). 가운데 3 이 사라져요')
ok(tuple(S.shape) == my_S_shape == (3, 3), '(3,2) @ (2,3) = (3,3). 단어끼리 전부 비교한 표예요')
ok(np.allclose(S, [[4, 0, 2], [0, 4, 2], [2, 2, 2]]), 'S 의 값은 [[4,0,2],[0,4,2],[2,2,2]]')
ok(head_dim == 64, 'd = 512 를 h = 8 로 나누면 헤드마다 64 칸')

## 4. 로그가 곱을 합으로 바꿔요

기초 4단원. $\exp$ 는 어떤 수든 양수로 바꿔 줘요. **로그(Logarithm)** 는 그 반대이고, 이 과목에서 $\log$ 라고만 적혀 있으면 **자연로그(Natural Logarithm)** 예요.

오늘의 핵심 한 줄은 $\log(xy) = \log x + \log y$ 예요. 확률을 계속 곱하면 0 으로 뭉개지지만, 로그를 씌우면 더하기가 되어 멀쩡해요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. exp(0) 은 1, exp(1) 은 약 2.718, exp(2) 는 약 7.389 예요
2. 곱한 뒤 로그를 씌운 값과, 따로 로그를 씌워 더한 값이 같은지 봐요
3. 단어 20개 문장에서 단어마다 확률이 0.1 이면 전체는 0.1 을 20번 곱한 값이에요
4. 로그로 바꾸면 ln 0.1 을 20번 더하는 것이 돼요. 곱하기가 더하기가 됐어요
5. 퍼플렉서티는 평균 손실을 exp 에 넣은 값이에요. 후보 4개에 0.25 씩 주면 4 가 나와야 해요

### 의사코드 (한 줄이 파이썬 한 줄이 돼요)

```
exp 값 세 개를 구한다
log(0.5) 더하기 log(0.25) 와 log(0.5 곱하기 0.25) 를 견준다
문장 로그 확률 = log(0.1) 곱하기 20
손실 = 정답 확률의 음의 로그
퍼플렉서티 = exp(손실)
```

> **나중에 여기서 만나요** N3 p.41-42 의 **퍼플렉서티** 가 바로 $\exp$(교차 엔트로피 손실) 이에요. N2 p.30-31 에서 word2vec 의 우도는 확률의 곱이고 로그를 씌워 합으로 만든 뒤 손실로 써요. N4 p.13 의 **빔 서치** 도 로그 확률을 더해서 점수를 매기고 길이로 나눠요.

In [ ]:
e0, e1, e2 = math.exp(0), math.exp(1), math.exp(2)

log_prod = math.log(0.5 * 0.25)
log_sum = None      # TODO: math.log(0.5) 와 math.log(0.25) 를 더하기

logp = None         # TODO: 확률 0.1 짜리 단어 20개. math.log(0.1) 에 20 을 곱해요

loss = -math.log(0.25)
ppl = None          # TODO: 손실을 exp 에 넣기

print(round(e2, 3), round(log_sum, 3), round(logp, 2), round(ppl, 4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(e0 - 1.0) < 1e-12 and abs(e2 - 7.389) < 1e-3, 'exp(0) 은 1, exp(2) 는 약 7.389')
ok(abs(log_sum - log_prod) < 1e-12, 'log(xy) = log x + log y. 곱이 합이 돼요')
ok(abs(log_sum - (-2.079)) < 1e-3, '두 값 모두 약 -2.079')
ok(abs(logp - (-46.05)) < 0.01, '0.1 을 20번 곱한 값은 로그로 -46.05. 그냥 곱하면 0 으로 뭉개져요')
ok(abs(ppl - 4.0) < 1e-9, '후보 4개에 0.25 씩 준 모델의 퍼플렉서티는 4')

## 5. 말뭉치에서 세어서 확률 구하기

기초 5단원. **확률(Probability)** 은 일어난 횟수를 전체 횟수로 나눈 값이에요. 조건을 붙이면 세는 범위가 그 안으로 좁아져요. 이것이 **조건부 확률(Conditional Probability)** 이에요.

`Counter` 는 목록에 무엇이 몇 번 나왔는지 세 주는 도구예요. 여기 말뭉치에서는 '나는' 과 '밥을' 이 문장 끝에 오는 일이 없어서 분모를 그냥 `uni[prev]` 로 써도 돼요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 단어를 모두 세요. 4문장에 3단어씩이니 12개예요
2. 단어 하나의 확률은 그 단어 횟수를 12 로 나눈 값이에요
3. 조건부 확률은 분모가 좁아져요. '밥을' 뒤를 볼 때 분모는 12 가 아니라 3 이에요
4. 붙어 있는 두 단어를 세려면 문장마다 (i 번째, i+1 번째) 짝을 만들어요
5. 문장 확률은 곱의 규칙으로 한 단어씩 곱해 나가요

> **나중에 여기서 만나요** N3 p.35-38 의 **엔그램 언어 모델** 이 바로 이 세어서 나누기예요. 여기서 `zero` 가 0 으로 나오는 것이 N3 p.39 의 희소성 문제이고, 그 때문에 N3 p.43 부터 신경망 언어 모델로 넘어가요. N4 p.10 의 디코더도 원문이 주어졌을 때의 조건부 확률 언어 모델이에요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
uni = 단어마다 몇 번 나왔는지 센다
bi = 붙어 있는 두 단어 짝이 몇 번 나왔는지 센다
total = uni 의 값을 모두 더한다
p_word(w) = uni[w] 를 total 로 나눈다
p_next(다음, 앞) = bi[(앞, 다음)] 을 uni[앞] 으로 나눈다
문장 확률 = p_word(첫 단어) 곱하기 p_next(...) 곱하기 p_next(...)
```

In [ ]:
corpus = [['나는', '밥을', '먹었다'],
          ['나는', '빵을', '먹었다'],
          ['나는', '밥을', '지었다'],
          ['너는', '밥을', '먹었다']]

uni = Counter(w for s in corpus for w in s)
bi = Counter((s[i], s[i + 1]) for s in corpus for i in range(len(s) - 1))

total = None                # TODO: 단어가 모두 몇 개인지. sum(uni.values())

def p_word(w):
    return None             # TODO: uni[w] 를 total 로 나누기

def p_next(nxt, prev):
    return None             # TODO: bi[(prev, nxt)] 를 uni[prev] 로 나누기

sent_p = None               # TODO: p_word('나는') 과 p_next('밥을', '나는') 과 0.5 를 모두 곱하기
zero = p_next('빵을', '밥을')

print(total, round(p_word('빵을'), 4), round(p_next('먹었다', '밥을'), 4))
print(round(sent_p, 4), zero)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(total == 12, '4문장에 3단어씩이라 모두 12개')
ok(abs(p_word('빵을') - 1 / 12) < 1e-12, '빵을 은 12개 중 1번이라 1/12, 약 0.083')
ok(abs(p_next('먹었다', '밥을') - 2 / 3) < 1e-12, '밥을 3번 중 먹었다가 2번이라 0.667')
ok(abs(p_next('지었다', '밥을') - 1 / 3) < 1e-12, '밥을 3번 중 지었다가 1번. 둘을 더하면 정확히 1 이에요')
ok(abs(sent_p - 1 / 12) < 1e-12, '0.25 x 0.667 x 0.5 = 1/12, 약 0.083')
ok(zero == 0, '밥을 빵을 은 한 번도 안 나와서 0. 이게 N3 p.39 의 희소성 문제예요')

## 6. 소프트맥스로 확률을 만들고 교차 엔트로피로 벌점을 매기기

기초 6단원. **소프트맥스(Softmax)** 는 두 줄이에요. exp 를 씌우고, 전체 합으로 나눠요. 모든 점수에서 최댓값을 똑같이 빼도 답이 변하지 않아서, 그렇게 하면 큰 수에서 넘치는 일을 막을 수 있어요.

**교차 엔트로피(Cross-Entropy)** 는 정답 자리에 준 확률의 **음의 로그(Negative Log)** 예요. 정답에 자신 있으면 벌점이 작고, 자신 있게 틀리면 벌점이 커요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 최댓값을 빼요. 결과는 안 변하고 넘침만 막아요
2. 각 점수에 exp 를 씌워요
3. 전체 합으로 나눠요. 여기까지가 소프트맥스예요
4. 교차 엔트로피는 정답 자리 확률 하나만 보고 음의 로그를 씌워요
5. 확인: 점수 2, 1, 0 은 확률 0.665, 0.245, 0.090 이 되고 벌점은 0.408 과 2.408 이에요

> **나중에 여기서 만나요** N2 p.32-33 에서 내적 점수를 소프트맥스로 확률로 바꾸는 것이 word2vec 의 마지막 조각이에요. N4 p.22 의 어텐션 2단계가 바로 이 `softmax` 이고, 교수님이 어텐션에서 가장 중요하다고 한 네 단계 중 하나예요. N4 p.38 은 점수가 너무 크면 이 분포가 뾰족해져서 기울기가 사라진다고 말해요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
softmax(z):
  z 에서 z 의 최댓값을 뺀다
  각 칸에 exp 를 씌운다
  전체 합으로 나눠서 돌려준다
cross_entropy(p, y):
  정답 자리 확률 p[y] 에 로그를 씌우고 마이너스를 붙인다
```

In [ ]:
def softmax(z):
    z = z - z.max()     # 최댓값 빼기. 답은 그대로고 넘침만 막아요
    e = None            # TODO: exp 씌우기 (np.exp)
    return None         # TODO: 전체 합으로 나누기 (e.sum())

def cross_entropy(p, y):
    return None         # TODO: 정답 자리 확률의 음의 로그 (np.log 사용)

z = np.array([2.0, 1.0, 0.0])
p = softmax(z)
loss_A = cross_entropy(p, 0)
loss_C = cross_entropy(p, 2)

print(p.round(3), round(float(loss_A), 3), round(float(loss_C), 3))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(p.sum() - 1.0) < 1e-12, '확률을 다 더하면 정확히 1')
ok(np.allclose(p, [0.665, 0.245, 0.090], atol=1e-3), '점수 2, 1, 0 이 확률 0.665, 0.245, 0.090 이 돼요')
ok(np.allclose(softmax(z + 100), p), '모든 점수에 같은 수를 더해도 결과가 같아요. 점수 차이만 중요해요')
ok(abs(float(loss_A) - 0.408) < 1e-3, '정답이 A 면 벌점 0.408')
ok(abs(float(loss_C) - 2.408) < 1e-3, '정답이 C 면 벌점 2.408. 자신 있게 틀리면 크게 혼나요')
ok(abs(float(loss_C - loss_A) - 2.0) < 1e-9, '벌점 차이 2 는 점수 차이 2 빼기 0 과 정확히 같아요')

## 7. 살짝 움직여 기울기 재기, 그리고 연쇄 법칙

기초 7단원. **미분(Derivative)** 은 x 를 아주 조금 움직였을 때 y 가 바뀌는 비율이에요. 그래서 `(f(x + h) - f(x)) / h` 에서 h 를 작게 줄이면 미분 값에 다가가요.

**연쇄 법칙(Chain Rule)** 은 단계마다의 기울기를 곱해서 잇는 규칙이에요. **편미분(Partial Derivative)** 은 하나만 움직이고 나머지는 상수처럼 두는 것이에요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 기울기 재기는 세로 변화를 가로 변화로 나누기예요
2. h 를 0.01 로 하면 6.01, 더 작게 하면 6 에 다가가요
3. 미분 규칙 네 개만 쓰면 돼요. 상수는 0, x제곱은 2x, x세제곱은 3x제곱, 상수배와 합은 그대로
4. 연쇄 법칙은 dy/du 와 du/dx 를 각각 구해서 곱하기예요
5. 편미분은 한 변수만 움직여요. 나머지는 숫자처럼 취급해요

> **나중에 여기서 만나요** N3 p.18-24 가 연쇄 법칙과 야코비안이고, 교수님이 3주차에 연쇄 법칙이 가장 중요하다고 했어요. N3 p.30 의 더하기, max, 곱하기 노드가 기울기를 흘리는 방식도 같은 규칙이에요. 무엇보다 과제 2(N3 p.57)가 word2vec 기울기를 손으로 유도하는 문제인데, 그 핵심이 소프트맥스에 이 연쇄 법칙을 쓰는 것이에요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
slope(f, x, h) = (f(x 더하기 h) 빼기 f(x)) 를 h 로 나눈다
g 프라임 = 10x 더하기 3x제곱 에 x = 2 를 넣는다
u = 3x 더하기 1 을 먼저 계산한다
dy/du = 2u, du/dx = 3
dy/dx = dy/du 곱하기 du/dx
```

In [ ]:
def slope(f, x, hstep):
    return None        # TODO: (f(x + hstep) - f(x)) / hstep

sq = lambda t: t * t
near = slope(sq, 3.0, 0.01)

g_prime_at_2 = None    # TODO: g(t) = 5t^2 + t^3 이면 g'(t) = 10t + 3t^2. t = 2 를 넣은 값

u = 3 * 1 + 1
dy_du = None           # TODO: y = u^2 이니 2u. u 는 위에서 구했어요
du_dx = None           # TODO: u = 3x + 1 을 x 로 미분한 값
dy_dx = dy_du * du_dx

dF_dx = 2 * 2 + 3 * 1  # F = x^2 + 3xy 를 x 로 편미분하면 2x + 3y, (2, 1) 을 넣었어요
dF_dy = None           # TODO: F 를 y 로 편미분하면 3x. (2, 1) 에서의 값

print(near, g_prime_at_2, dy_dx, dF_dx, dF_dy)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(near - 6.01) < 1e-6, 'x = 3 에서 0.01 만큼 움직이면 6.01 이 나와요')
ok(abs(slope(sq, 3.0, 1e-7) - 6.0) < 1e-5, '움직이는 양을 아주 작게 줄이면 미분 값 6 에 다가가요')
ok(g_prime_at_2 == 32, '10x2 = 20, 3x4 = 12, 더하면 32')
ok(dy_du == 8 and du_dx == 3 and dy_dx == 24, '2u x 3 = 8 x 3 = 24. 펼쳐서 18x + 6 에 x = 1 을 넣어도 24 예요')
ok(dF_dx == 7 and dF_dy == 6, '편미분은 하나만 움직여요. 7 과 6 이에요')

## 8. 경사 하강법 한 걸음씩 내려가기

기초 8단원. **경사 하강법(Gradient Descent)** 한 걸음은 `w <- w - alpha * 기울기` 예요. 기울기는 올라갈 방향이라서 빼요. `alpha` 가 **학습률(Learning Rate)**, 곧 보폭이에요.

여기서는 $L(w) = w^2$ 라서 기울기가 $2w$ 예요. 보폭을 1.5 로 키우면 내려가기는커녕 튕겨 나가는 것도 같이 봐요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 기울기는 2w 예요. 함수 한 줄로 만들어 두면 반복이 쉬워요
2. 한 걸음은 지금 w 에서 alpha 곱하기 기울기를 빼는 것이에요
3. 세 걸음을 걸으려면 같은 일을 세 번 반복해요. for 문이 그 일을 해 줘요
4. 걸음마다 w 를 기록해 두면 4, 2, 1, 0.5 로 줄어드는 게 보여요
5. 보폭 1.5 로 같은 자리에서 시작하면 -8, 16 으로 커져요. 보폭이 크면 발산해요

> **나중에 여기서 만나요** N2 p.35 가 경사 하강법과 보폭 $\alpha$, N2 p.36 이 미니배치를 쓰는 **확률적 경사 하강법** 이에요. N3 p.17 의 신경망 학습도 같은 방식이고, 3주차 실습 N3L p.12 의 학습 루프가 이 `for` 문 그대로예요. 과제 2 에서 스킵그램을 구현할 때도 이 걸음을 돌려요.

### 의사코드 먼저

코드를 치기 전에 아래 줄을 종이에 옮겨 적어요. 한 줄이 파이썬 한 줄이 돼요.

```
step(w, alpha):
  기울기 = 2 곱하기 w
  새 w = w 빼기 alpha 곱하기 기울기
w 를 4 에서 시작해 같은 일을 세 번 반복하고 기록한다
보폭만 1.5 로 바꿔서 두 번 반복하고 기록한다
한 에폭의 걸음 수 = 데이터 개수 나누기 미니배치 크기
```

In [ ]:
def step(w, alpha):
    grad = None        # TODO: L(w) = w^2 의 기울기, 곧 2w
    return None        # TODO: w 에서 alpha 곱하기 grad 를 빼기

w = 4.0
path = [w]
for _ in range(3):
    w = step(w, 0.25)
    path.append(w)

big = [4.0]
for _ in range(2):
    big.append(step(big[-1], 1.5))

n_data, batch = 1000, 100
steps_per_epoch = None   # TODO: 한 에폭에 몇 걸음인지 (// 사용)

print(path)
print(big, steps_per_epoch)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(path == [4.0, 2.0, 1.0, 0.5], 'w 가 4, 2, 1, 0.5 로 줄어요. 슬라이드 손계산 2 와 같아요')
ok([v * v for v in path] == [16.0, 4.0, 1.0, 0.25], '손실은 16, 4, 1, 0.25 로 줄어요')
ok(big == [4.0, -8.0, 16.0], '보폭 1.5 면 -8, 16 으로 튕겨 나가요. 손실은 64, 256 이에요')
ok(steps_per_epoch == 10, '1000 을 100 으로 나누면 한 에폭에 10 걸음, 3 에폭이면 30 걸음')

## 9. 뉴런 한 줄과 순전파 한 번

기초 9단원. **뉴런(Neuron)** 하나는 **가중합(Weighted Sum)** 에 **편향 항(Bias Term)** 을 더하고 **활성화 함수(Activation Function)** 로 한 번 구부린 것이에요.

여기서는 뉴런 셋을 한 층으로 모아 한 번에 계산하고, **렐루(ReLU)** 를 지나 마지막 한 칸으로 줄여요. 입력에서 출력까지 앞으로 한 번 계산하는 것이 **순전파(Forward Pass)** 예요. 마지막 칸에서는 활성화 함수가 없으면 층을 쌓아도 한 층이라는 것도 확인해요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 층 하나를 행렬 곱 한 번으로 써요. W1 은 (3, 2), x 는 (2,) 라서 결과는 (3,)
2. 거기에 편향 항 b1 을 더해요. 칸마다 하나씩 더해져요
3. 렐루는 음수를 0 으로 눌러요. -1 이 0 이 되고 5 와 2 는 그대로예요
4. 둘째 층은 칸이 하나뿐이라 내적 한 번에 편향을 더하면 끝이에요
5. 마지막은 시그모이드라 0 과 1 사이 값, 곧 확률처럼 읽을 수 있어요
6. 직선 두 개를 이으면 y = 2t + 1 다음 z = 3y - 2 가 결국 z = 6t + 1 한 줄이에요

> **나중에 여기서 만나요** N3 p.9-10 이 뉴런과 활성화 함수이고, 교수님이 렐루가 딥러닝에서 가장 중요한 함수 중 하나라고 했어요. N3 p.11 이 여기 마지막 칸에서 확인한 것, 곧 비선형성이 없으면 탑 전체가 한 층으로 무너진다는 내용이에요. N3 p.44 의 고정 윈도우 신경망 언어 모델이 정확히 이 구조이고, N4 p.42 의 트랜스포머 위치별 피드포워드 신경망도 2층짜리 같은 구조예요.

### 의사코드 먼저

이번에는 순서를 스스로 세워 봐요. 위 **생각 순서**를 보고 의사코드를 서너 줄로 적은 다음에 코드를 쳐요. 모범 의사코드는 맨 아래 정답 모음에 있어요.

In [ ]:
def relu(v):
    return np.maximum(0.0, v)

def sigmoid(t):
    return 1.0 / (1.0 + math.exp(-t))

x = np.array([1.0, 2.0])
W1 = np.array([[1.0, -1.0],
               [0.0, 2.0],
               [1.0, 1.0]])
b1 = np.array([0.0, 1.0, -1.0])
w2 = np.array([1.0, -1.0, 2.0])
b2 = 0.5

z1 = None        # TODO: W1 과 x 의 행렬 곱에 b1 을 더하기
h1 = None        # TODO: z1 을 렐루에 통과시키기
z2 = None        # TODO: w2 와 h1 의 내적에 b2 를 더하기 (w2 @ h1)
out = sigmoid(float(z2))

lin = lambda t: 3 * (2 * t + 1) - 2
collapsed = None  # TODO: 위 두 직선을 하나로 합친 식. lambda t: 로 시작해요

print(z1, h1, round(float(z2), 4), round(out, 4))
print(lin(2), collapsed(2))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(np.allclose(z1, [-1.0, 5.0, 2.0]), '가중합에 편향 항을 더하면 (-1, 5, 2)')
ok(np.allclose(h1, [0.0, 5.0, 2.0]), '렐루가 음수 -1 을 0 으로 눌러요')
ok(abs(float(z2) - (-0.5)) < 1e-12, '0 - 5 + 4 = -1 에 편향 0.5 를 더해 -0.5')
ok(abs(out - 0.3775) < 1e-4, '시그모이드에 넣으면 약 0.3775, 곧 약 38 퍼센트')
ok(all(lin(t) == collapsed(t) for t in (0, 1, 2, 5)), '직선 두 층은 결국 6t + 1 한 층이에요. 그래서 비선형성이 필요해요')

## 10. 실습 노트북에 나오는 파이썬과 PyTorch

기초 10단원. 실습 탭 코드에 계속 나오는 것만 모았어요. **리스트(List)** 의 칸 번호는 0 부터이고, `Counter` 는 무엇이 몇 번 나왔는지 세 줘요.

골뱅이 `@` 는 같은 자리끼리 곱하기가 아니라 **행렬 곱(Matrix Multiplication)** 이에요. `requires_grad=True` 로 만든 텐서에 `backward()` 를 부르면 **자동 미분(Automatic Differentiation (Autodiff))** 이 기울기를 대신 구해 줘요. 손으로 구한 값과 같은지 꼭 견줘 봐요.

### 생각 순서 (코드 치기 전에 말로 읊어요)

1. 칸 번호는 0 부터라서 h[1] 이 둘째 칸이에요
2. Counter 에 목록을 넣으면 무엇이 몇 번인지 세 줘요
3. W 는 (3, 2), x 는 (2,) 라서 골뱅이 곱의 결과는 (3,) 이에요
4. f = x세제곱 + 2x 를 손으로 미분하면 3x제곱 + 2 예요
5. x = 2 를 넣으면 3 곱하기 4 더하기 2 로 14 예요. autograd 도 14 가 나와야 해요

> **나중에 여기서 만나요** 2주차 실습 N2L p.9-10 이 딕셔너리로 만든 말뭉치와 Counter 로 짝 세기, 3주차 실습 N3L p.3 이 shape 찍어 보기와 골뱅이, p.4 가 requires_grad 와 backward 예요. 저장된 출력도 14.0 이에요. N3L p.12 의 학습 루프가 그 기울기로 8번 문제의 걸음을 걷고, 4주차 실습 N4L 은 같은 문법으로 쿼리, 키, 밸류를 만들어요.

### 의사코드 먼저

이번에는 순서를 스스로 세워 봐요. 위 **생각 순서**를 보고 의사코드를 서너 줄로 적은 다음에 코드를 쳐요. 모범 의사코드는 맨 아래 정답 모음에 있어요.

In [ ]:
h = [0, 5, 2]
second = None       # TODO: 둘째 칸의 값. 칸 번호는 0 부터예요
length = None       # TODO: 칸이 몇 개인지

words = ['영화', '영화', '배우', '최고', '영화', '배우']
freq = None         # TODO: Counter 로 세기

W = torch.tensor([[1.0, -1.0],
                  [0.0, 2.0],
                  [1.0, 1.0]])
xt = torch.tensor([1.0, 2.0])
z = None            # TODO: 골뱅이로 행렬 곱

xg = torch.tensor(2.0, requires_grad=True)
f = xg ** 3 + 2 * xg
f.backward()
by_hand = None      # TODO: f' = 3x^2 + 2 에 x = 2 를 넣은 값

print(second, length, freq.most_common(2))
print(z, tuple(z.shape), xg.grad.item(), by_hand)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(second == 5 and length == 3, 'h[1] 은 5, len(h) 는 3. 칸 번호는 0 부터예요')
ok(freq.most_common(2) == [('영화', 3), ('배우', 2)], '영화 3번, 배우 2번이에요')
ok(tuple(z.shape) == (3,), '(3,2) 골뱅이 (2,) 는 (3,). 가운데 2 가 사라져요')
ok(torch.allclose(z, torch.tensor([-1.0, 4.0, 3.0])), 'z 는 (-1, 4, 3). 9번 문제의 가중합과 같은 숫자예요')
ok(by_hand == 14 and abs(xg.grad.item() - 14.0) < 1e-6, '손계산도 14, autograd 도 14. 실습 N3L p.4 의 저장된 출력과 같아요')

## 정답 코드

먼저 스스로 풀어 보고, 막혔을 때만 봐요.

**1. 벡터는 숫자 목록, 차원은 칸의 개수**

의사코드

```
a 와 b 를 리스트로 적는다
차원 = a 의 칸 개수
a + b = [첫 칸끼리 더한 값, 둘째 칸끼리 더한 값]
모텔 벡터와 호텔 벡터를 같은 자리끼리 곱해서 모두 더한다
```

```python
a = [2, 1]
b = [1, 3]

dim = len(a)
a_plus_b = [a[0] + b[0], a[1] + b[1]]
two_a = [2 * v for v in a]

vocab = ['개', '고양이', '모텔', '호텔', '사과']

def onehot(word):
    i = vocab.index(word)
    return [1 if k == i else 0 for k in range(len(vocab))]

motel = onehot('모텔')
hotel = onehot('호텔')
same = sum(x * y for x, y in zip(motel, hotel))

print(dim, a_plus_b, two_a)
print(motel, hotel, same)
```

**2. 내적, 길이, 코사인 유사도를 함수로**

의사코드

```
dot(u, v) = 자리마다 u 와 v 를 곱한 값을 모두 더한다
norm(u) = dot(u, u) 에 루트를 씌운다
cosine(u, v) = dot(u, v) 를 norm(u) 곱하기 norm(v) 로 나눈다
점수 = 후보마다 dot(q, 후보)
가장 큰 점수의 자리 번호를 찾는다
```

```python
def dot(u, v):
    return sum(x * y for x, y in zip(u, v))

def norm(u):
    return math.sqrt(dot(u, u))

def cosine(u, v):
    return dot(u, v) / (norm(u) * norm(v))

print(dot([3, 2], [1, 4]), norm([3, 4]), round(cosine([3, 4], [4, 3]), 4))

q = [1, 2]
keys = [[2, 0], [0, 3], [1, 1]]
scores = [dot(q, k) for k in keys]
best = scores.index(max(scores))
print(scores, best)
```

**3. shape 을 먼저 적고 나서 곱하기**

의사코드

```
A @ x 의 모양을 먼저 종이에 적는다: (2,3) 과 (3,) 이니 (2,)
실제로 곱해서 값을 본다
Q 와 K 의 모양을 적는다: 둘 다 (3,2)
K 를 전치해서 (2,3) 으로 만들고 Q 와 곱한다
헤드 하나의 칸 수 = 전체 칸 수 나누기 헤드 수
```

```python
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
x = np.array([1.0, 0.0, 2.0])

Ax = A @ x
my_Ax_shape = (2,)

Q = np.array([[2.0, 0.0],
              [0.0, 2.0],
              [1.0, 1.0]])
K = Q.copy()
S = Q @ K.T
my_S_shape = (3, 3)

d, h = 512, 8
head_dim = d // h

print(Ax, tuple(S.shape), head_dim)
```

**4. 로그가 곱을 합으로 바꿔요**

의사코드

```
exp 값 세 개를 구한다
log(0.5) 더하기 log(0.25) 와 log(0.5 곱하기 0.25) 를 견준다
문장 로그 확률 = log(0.1) 곱하기 20
손실 = 정답 확률의 음의 로그
퍼플렉서티 = exp(손실)
```

```python
e0, e1, e2 = math.exp(0), math.exp(1), math.exp(2)

log_prod = math.log(0.5 * 0.25)
log_sum = math.log(0.5) + math.log(0.25)

logp = 20 * math.log(0.1)

loss = -math.log(0.25)
ppl = math.exp(loss)

print(round(e2, 3), round(log_sum, 3), round(logp, 2), round(ppl, 4))
```

**5. 말뭉치에서 세어서 확률 구하기**

의사코드

```
uni = 단어마다 몇 번 나왔는지 센다
bi = 붙어 있는 두 단어 짝이 몇 번 나왔는지 센다
total = uni 의 값을 모두 더한다
p_word(w) = uni[w] 를 total 로 나눈다
p_next(다음, 앞) = bi[(앞, 다음)] 을 uni[앞] 으로 나눈다
문장 확률 = p_word(첫 단어) 곱하기 p_next(...) 곱하기 p_next(...)
```

```python
corpus = [['나는', '밥을', '먹었다'],
          ['나는', '빵을', '먹었다'],
          ['나는', '밥을', '지었다'],
          ['너는', '밥을', '먹었다']]

uni = Counter(w for s in corpus for w in s)
bi = Counter((s[i], s[i + 1]) for s in corpus for i in range(len(s) - 1))

total = sum(uni.values())

def p_word(w):
    return uni[w] / total

def p_next(nxt, prev):
    return bi[(prev, nxt)] / uni[prev]

sent_p = p_word('나는') * p_next('밥을', '나는') * 0.5
zero = p_next('빵을', '밥을')

print(total, round(p_word('빵을'), 4), round(p_next('먹었다', '밥을'), 4))
print(round(sent_p, 4), zero)
```

**6. 소프트맥스로 확률을 만들고 교차 엔트로피로 벌점을 매기기**

의사코드

```
softmax(z):
  z 에서 z 의 최댓값을 뺀다
  각 칸에 exp 를 씌운다
  전체 합으로 나눠서 돌려준다
cross_entropy(p, y):
  정답 자리 확률 p[y] 에 로그를 씌우고 마이너스를 붙인다
```

```python
def softmax(z):
    z = z - z.max()     # 최댓값 빼기. 답은 그대로고 넘침만 막아요
    e = np.exp(z)
    return e / e.sum()

def cross_entropy(p, y):
    return -np.log(p[y])

z = np.array([2.0, 1.0, 0.0])
p = softmax(z)
loss_A = cross_entropy(p, 0)
loss_C = cross_entropy(p, 2)

print(p.round(3), round(float(loss_A), 3), round(float(loss_C), 3))
```

**7. 살짝 움직여 기울기 재기, 그리고 연쇄 법칙**

의사코드

```
slope(f, x, h) = (f(x 더하기 h) 빼기 f(x)) 를 h 로 나눈다
g 프라임 = 10x 더하기 3x제곱 에 x = 2 를 넣는다
u = 3x 더하기 1 을 먼저 계산한다
dy/du = 2u, du/dx = 3
dy/dx = dy/du 곱하기 du/dx
```

```python
def slope(f, x, hstep):
    return (f(x + hstep) - f(x)) / hstep

sq = lambda t: t * t
near = slope(sq, 3.0, 0.01)

g_prime_at_2 = 10 * 2 + 3 * 2 ** 2

u = 3 * 1 + 1
dy_du = 2 * u
du_dx = 3
dy_dx = dy_du * du_dx

dF_dx = 2 * 2 + 3 * 1
dF_dy = 3 * 2

print(near, g_prime_at_2, dy_dx, dF_dx, dF_dy)
```

**8. 경사 하강법 한 걸음씩 내려가기**

의사코드

```
step(w, alpha):
  기울기 = 2 곱하기 w
  새 w = w 빼기 alpha 곱하기 기울기
w 를 4 에서 시작해 같은 일을 세 번 반복하고 기록한다
보폭만 1.5 로 바꿔서 두 번 반복하고 기록한다
한 에폭의 걸음 수 = 데이터 개수 나누기 미니배치 크기
```

```python
def step(w, alpha):
    grad = 2 * w
    return w - alpha * grad

w = 4.0
path = [w]
for _ in range(3):
    w = step(w, 0.25)
    path.append(w)

big = [4.0]
for _ in range(2):
    big.append(step(big[-1], 1.5))

n_data, batch = 1000, 100
steps_per_epoch = n_data // batch

print(path)
print(big, steps_per_epoch)
```

**9. 뉴런 한 줄과 순전파 한 번**

의사코드

```
z1 = W1 과 x 를 행렬 곱 하고 b1 을 더한다
h1 = z1 을 렐루에 통과시킨다
z2 = w2 와 h1 의 내적에 b2 를 더한다
out = z2 를 시그모이드에 넣는다
직선 두 개를 하나로 합치면 6t 더하기 1 이라고 적는다
```

```python
def relu(v):
    return np.maximum(0.0, v)

def sigmoid(t):
    return 1.0 / (1.0 + math.exp(-t))

x = np.array([1.0, 2.0])
W1 = np.array([[1.0, -1.0],
               [0.0, 2.0],
               [1.0, 1.0]])
b1 = np.array([0.0, 1.0, -1.0])
w2 = np.array([1.0, -1.0, 2.0])
b2 = 0.5

z1 = W1 @ x + b1
h1 = relu(z1)
z2 = w2 @ h1 + b2
out = sigmoid(float(z2))

lin = lambda t: 3 * (2 * t + 1) - 2
collapsed = lambda t: 6 * t + 1

print(z1, h1, round(float(z2), 4), round(out, 4))
print(lin(2), collapsed(2))
```

**10. 실습 노트북에 나오는 파이썬과 PyTorch**

의사코드

```
h 의 둘째 칸과 칸 개수를 읽는다
Counter 로 단어를 센다
z = W 골뱅이 x
x 를 requires_grad 로 만들고 f 를 계산한 뒤 backward 를 부른다
손으로 미분한 3x제곱 더하기 2 에 x = 2 를 넣어 견준다
```

```python
h = [0, 5, 2]
second = h[1]
length = len(h)

words = ['영화', '영화', '배우', '최고', '영화', '배우']
freq = Counter(words)

W = torch.tensor([[1.0, -1.0],
                  [0.0, 2.0],
                  [1.0, 1.0]])
xt = torch.tensor([1.0, 2.0])
z = W @ xt

xg = torch.tensor(2.0, requires_grad=True)
f = xg ** 3 + 2 * xg
f.backward()
by_hand = 3 * 2 ** 2 + 2

print(second, length, freq.most_common(2))
print(z, tuple(z.shape), xg.grad.item(), by_hand)
```